# E-commerce Analytics Visualization

This notebook demonstrates how to query and visualize the data from our dbt-transformed marts.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Connect to DuckDB warehouse
conn = duckdb.connect('../data/warehouse.duckdb')
print('Connected to DuckDB warehouse')

## 1. Customer Analytics

In [ ]:
# Customer segmentation distribution
df_segments = conn.execute("""
    SELECT 
        customer_segment,
        COUNT(*) as customer_count,
        ROUND(SUM(lifetime_value), 2) as total_ltv
    FROM mart_customer_metrics
    GROUP BY customer_segment
    ORDER BY total_ltv DESC
""").df()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Customer count by segment
ax1.bar(df_segments['customer_segment'], df_segments['customer_count'], color='steelblue')
ax1.set_title('Customer Count by Segment')
ax1.set_xlabel('Customer Segment')
ax1.set_ylabel('Number of Customers')
ax1.tick_params(axis='x', rotation=45)

# Revenue by segment
ax2.bar(df_segments['customer_segment'], df_segments['total_ltv'], color='coral')
ax2.set_title('Total Lifetime Value by Segment')
ax2.set_xlabel('Customer Segment')
ax2.set_ylabel('Lifetime Value ($)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Activity status distribution
df_activity = conn.execute("""
    SELECT 
        activity_status,
        COUNT(*) as customer_count
    FROM mart_customer_metrics
    GROUP BY activity_status
""").df()

plt.figure(figsize=(10, 6))
plt.pie(df_activity['customer_count'], labels=df_activity['activity_status'], 
        autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set2'))
plt.title('Customer Activity Status Distribution')
plt.show()

## 2. Product Performance

In [ ]:
# Top 10 products by revenue
df_top_products = conn.execute("""
    SELECT 
        product_name,
        category,
        total_revenue,
        profit_margin_pct
    FROM mart_product_performance
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

plt.figure(figsize=(12, 6))
plt.barh(df_top_products['product_name'], df_top_products['total_revenue'], color='teal')
plt.xlabel('Total Revenue ($)')
plt.title('Top 10 Products by Revenue')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Category performance
df_categories = conn.execute("""
    SELECT 
        category,
        ROUND(SUM(total_revenue), 2) as revenue,
        ROUND(AVG(profit_margin_pct), 2) as avg_margin
    FROM mart_product_performance
    GROUP BY category
    ORDER BY revenue DESC
""").df()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Revenue by category
ax1.bar(df_categories['category'], df_categories['revenue'], color='mediumseagreen')
ax1.set_title('Revenue by Category')
ax1.set_xlabel('Category')
ax1.set_ylabel('Revenue ($)')
ax1.tick_params(axis='x', rotation=45)

# Profit margin by category
ax2.bar(df_categories['category'], df_categories['avg_margin'], color='indianred')
ax2.set_title('Average Profit Margin by Category')
ax2.set_xlabel('Category')
ax2.set_ylabel('Profit Margin (%)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Revenue Trends

In [ ]:
# Daily revenue with moving average
df_revenue = conn.execute("""
    SELECT 
        date,
        daily_revenue,
        revenue_7day_ma
    FROM mart_daily_revenue
    ORDER BY date
""").df()

plt.figure(figsize=(14, 6))
plt.plot(df_revenue['date'], df_revenue['daily_revenue'], alpha=0.4, label='Daily Revenue')
plt.plot(df_revenue['date'], df_revenue['revenue_7day_ma'], linewidth=2, label='7-Day Moving Average')
plt.xlabel('Date')
plt.ylabel('Revenue ($)')
plt.title('Daily Revenue Trend')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly revenue trend
df_monthly = conn.execute("""
    SELECT 
        order_month,
        ROUND(SUM(daily_revenue), 2) as monthly_revenue,
        SUM(order_count) as total_orders
    FROM mart_daily_revenue
    GROUP BY order_month
    ORDER BY order_month
""").df()

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.bar(df_monthly['order_month'], df_monthly['monthly_revenue'], color='dodgerblue', alpha=0.7)
ax1.set_xlabel('Month')
ax1.set_ylabel('Revenue ($)', color='dodgerblue')
ax1.tick_params(axis='y', labelcolor='dodgerblue')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(df_monthly['order_month'], df_monthly['total_orders'], color='red', marker='o', linewidth=2)
ax2.set_ylabel('Number of Orders', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Monthly Revenue and Order Count')
plt.tight_layout()
plt.show()

## 4. Summary Statistics

In [ ]:
# Overall business metrics
summary = conn.execute("""
    SELECT 
        (SELECT COUNT(*) FROM mart_customer_metrics) as total_customers,
        (SELECT COUNT(*) FROM mart_product_performance) as total_products,
        (SELECT ROUND(SUM(daily_revenue), 2) FROM mart_daily_revenue) as total_revenue,
        (SELECT ROUND(AVG(lifetime_value), 2) FROM mart_customer_metrics) as avg_customer_ltv,
        (SELECT ROUND(AVG(avg_order_value), 2) FROM mart_daily_revenue) as avg_order_value
""").df()

print('=== Business Summary ===')
print(f"Total Customers: {summary['total_customers'][0]:,}")
print(f"Total Products: {summary['total_products'][0]:,}")
print(f"Total Revenue: ${summary['total_revenue'][0]:,.2f}")
print(f"Average Customer LTV: ${summary['avg_customer_ltv'][0]:,.2f}")
print(f"Average Order Value: ${summary['avg_order_value'][0]:,.2f}")

In [ ]:
# Close connection
conn.close()